# Macro Stress Testing

Build a **grounded, self-verifying** agent that links a payments processor's **portfolio volume and credit health** to the **macro backdrop** (CPI, unemployment, Fed funds, PCE, consumer sentiment) — and scaffolds **what-if stress sims** via elasticity betas.

| Step | What you build |
|---|---|
| 1 | Explore the benchmark (`MACRO_STRESS_BENCHMARK`) |
| 2 | Query the **semantic view** (Cortex Analyst) |
| 3 | Build a verified **fact pack** |
| 4 | **Grounded draft** under DATA RULES |
| 5 | **Verify** claims vs evidence |
| 6 | Persist the note |

**Prereq:** run `setup_synthetic.sql` (no listing) or `setup.sql` (real Cybersyn macro). Set the notebook role to one that can read `SNOWPACK_MACRO_STRESS.CORE`.

In [ ]:
SCHEMA = "SNOWPACK_MACRO_STRESS.CORE"
MODEL  = "claude-sonnet-4-5"
import json
from snowflake.snowpark.context import get_active_session
session = get_active_session()
mode = session.sql(f"select mode, note from {SCHEMA}.MACRO_STRESS_DATA_SOURCE_INFO").collect()[0]
print("data mode:", mode[0], "—", mode[1])

## Step 1 — Explore the benchmark

`MACRO_STRESS_BENCHMARK` = synthetic portfolio (volume, approval, delinquency, charge-off) by credit tier/product/month, joined to the monthly macro backdrop. Macro columns are non-additive (one value per month, repeated across tier/product).

In [ ]:
session.sql(f"""
  SELECT credit_tier, product, month_end,
         ROUND(transaction_volume_usd/1e6,1) AS volume_m,
         approval_rate, delinquency_rate_30d, charge_off_rate,
         unemployment_rate, fed_funds_rate, consumer_sentiment
  FROM {SCHEMA}.MACRO_STRESS_BENCHMARK
  WHERE month_end = (SELECT MAX(month_end) FROM {SCHEMA}.MACRO_STRESS_BENCHMARK)
  ORDER BY volume_m DESC
""").to_pandas()

## Step 2 — Cortex Analyst over the semantic view

`MACRO_STRESS_SV` lets Analyst answer NL questions like *"which tiers have the highest delinquency in 2026?"*. Direct `SEMANTIC_VIEW()` query below; the agent writes this for you from plain English. Macro metrics are AVG (non-additive).

In [ ]:
session.sql(f"""
  SELECT * FROM SEMANTIC_VIEW({SCHEMA}.MACRO_STRESS_SV
      METRICS total_volume, avg_delinquency_30d, avg_charge_off_rate, avg_unemployment
      DIMENSIONS credit_tier, report_year)
  WHERE report_year = 2026 ORDER BY avg_charge_off_rate DESC
""").to_pandas()

## Step 3 — Verified fact pack

Start-of-window and latest snapshots per credit tier/product (volume, approval, delinquency, charge-off) plus the macro backdrop at those months — the agent's only allowed numbers.

In [ ]:
fact_pack = session.sql(f"""
  WITH w AS (SELECT MIN(month_end) a, MAX(month_end) z FROM {SCHEMA}.MACRO_STRESS_BENCHMARK)
  SELECT ARRAY_AGG(OBJECT_CONSTRUCT(
           'credit_tier', s.credit_tier, 'product', s.product, 'month', s.month_end::string,
           'transaction_volume_usd', s.transaction_volume_usd, 'approval_rate', s.approval_rate,
           'delinquency_rate_30d', s.delinquency_rate_30d, 'charge_off_rate', s.charge_off_rate,
           'unemployment_rate', s.unemployment_rate, 'fed_funds_rate', s.fed_funds_rate,
           'consumer_sentiment', s.consumer_sentiment, 'is_holiday_season', s.is_holiday_season))
  FROM {SCHEMA}.MACRO_STRESS_BENCHMARK s JOIN w ON s.month_end IN (w.a, w.z)
""").collect()[0][0]
print(fact_pack[:700], '...')

## Step 4 — Grounded draft (DATA RULES)

The model may use ONLY the fact pack — no invented numbers, no hedging.

In [ ]:
DATA_RULES = (
    "You are a payments risk & finance analyst. Write a concise portfolio stress note using ONLY the FACTS JSON.\n"
    "DATA RULES: use only numbers in FACTS, copy exactly; rates are decimals (0.04 = 4%); unemployment and fed_funds are in percentage points; "
    "dollar fields are USD, express in $B/$M; macro values are a non-additive backdrop (do not sum); "
    "compute volume change as (latest - earliest) per tier/product from FACTS; no estimates, no hedging words, no outside benchmarks; missing = n/d.\n"
    "OUTPUT markdown starting with '# ', a one-line takeaway (how volume & credit health moved with the macro backdrop), a table "
    "(credit tier | product | volume start->now | approval | delinquency | charge-off), then 3 bullets (incl. holiday-season effect and the most downturn-sensitive tier), ending with '## Data Coverage' (sources + window)."
)
prompt = DATA_RULES + f"\n\nFACTS: {fact_pack}"
draft_md = session.sql("SELECT AI_COMPLETE(?, ?)", params=[MODEL, prompt]).collect()[0][0]
print(draft_md)

## Step 5 — Verify against the evidence

A second pass fact-checks every number against the fact pack and flags hedges — the quality gate.

In [ ]:
verify_prompt = (
    "You are a fact-checker. Compare the DRAFT against the FACTS JSON (the only allowed source). "
    "Return ONLY a valid JSON object (double quotes, no code fences, no prose) with keys: status (\"verified\" if every number "
    "and claim is supported by FACTS and there are no hedge words like likely/approximately/around/could, else \"needs_review\"), "
    "residual_hedges (integer), edits (array of short strings naming each unsupported or hedged claim).\n\n"
    f"FACTS: {fact_pack}\n\nDRAFT:\n{draft_md}"
)
raw = session.sql("SELECT AI_COMPLETE(?, ?)", params=[MODEL, verify_prompt]).collect()[0][0]
import re as _re
def _parse_verdict(x):
    t = x.strip()
    for _ in range(3):
        m = _re.match(r"^```[a-zA-Z]*\s*(.*?)\s*```$", t, _re.S)
        if m:
            t = m.group(1).strip()
        try:
            v = json.loads(t)
        except Exception:
            a, b = t.find("{"), t.rfind("}")
            if a >= 0 and b > a:
                try:
                    v = json.loads(t[a:b + 1])
                except Exception:
                    return None
            else:
                return None
        if isinstance(v, str):
            t = v.strip()
            continue
        return v
    return None
verdict = _parse_verdict(raw) or {"status": "needs_review", "residual_hedges": None, "edits": ["unparseable verifier output"]}
status = verdict.get("status", "needs_review")
edits = verdict.get("edits", []) or []
hedges = verdict.get("residual_hedges")
print(f"STATUS: {status.upper()}   residual_hedges={hedges}   edits={len(edits)}")
for e in edits:
    print(" -", e)

## Step 6 — Persist the note

In [ ]:
session.sql(
    f"INSERT INTO {SCHEMA}.MACRO_STRESS_REPORTS (subject, draft_md, report_md, status, edits_applied, residual_hedges) SELECT ?, ?, ?, ?, ?, ?",
    params=["Portfolio macro stress", draft_md, draft_md, status, len(edits), hedges]
).collect()
session.sql(f"SELECT run_id, subject, status, edits_applied, residual_hedges, created_at FROM {SCHEMA}.MACRO_STRESS_REPORTS ORDER BY created_at DESC LIMIT 5").to_pandas()

## Recap & extensions

You built the verified macro-stress loop on free data. `setup.sql` already created a **`MACRO_STRESS_AGENT`** — chat with it in Snowsight → AI & ML → Agents (*"which tiers are most sensitive to rising unemployment?"*).

**What-if sims:** `MACRO_STRESS_BETA` holds elasticities (e.g. subprime approval_rate beta vs Fed funds = -0.014/pp). Multiply a macro shock by the beta to model the portfolio impact.

**Next:** point `MACRO_STRESS_PORTFOLIO_MONTHLY` at your own authorization/settlement data to stress your real book against the macro cycle. The same grounded, self-verifying pattern reuses across other payments use cases.

**Try breaking the verifier:** add a made-up forecast (e.g., *"charge-offs likely double in a recession"*) to the draft and watch Step 5 flag it *Needs review*.